# V2 Market Leading Features

## 목적

V1 데이터에 시장 수급·가격 흐름·소비자 심리 지표를 추가하여
V2용 선행지표 Feature를 생성한다.

### 주요 데이터

#### 한국부동산원 R-ONE
- 아파트 매매수급동향
- 아파트 전세수급동향
- 매매가격지수
- 전세가격지수

#### 한국은행
- 주택가격전망CSI
- 소비자심리지수(CCSI)

### 기본 원칙

- 기존 `df_final_v1.parquet`는 수정하지 않는다.
- 월별 지표는 미래정보 누출을 방지하기 위해 기본적으로 `lag1`을 사용한다.
- R-ONE 가격지수는 최신월(2026-06)=100으로 리베이스되어 있어 **지수 수준값은 미래 상승폭 정보를 포함**한다.
  가격지수는 기준시점과 무관한 증감률·Momentum만 Feature로 사용한다.
- 원본 데이터는 `data/external/`에서 불러온다.
- 최종 결과는 `data/v2_5year_seoul/processed/`에 저장한다.

In [1]:
# ============================================================
# 프로젝트 경로 설정
# - 현재 노트북 위치와 관계없이 data 폴더가 있는 프로젝트 루트를 찾는다.
# - external: 외부기관 원본 데이터 경로
# - processed: 정제 완료 데이터 저장 경로
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "프로젝트 루트에서 data 폴더를 찾을 수 없습니다."
    )


PROJECT_DIR = find_project_root()

EXTERNAL_DIR = PROJECT_DIR / "data" / "external"

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "v2_5year_seoul"
    / "processed"
)

print("PROJECT_DIR:", PROJECT_DIR)
print("EXTERNAL_DIR:", EXTERNAL_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_DIR: C:\Users\uk\Apartment price
EXTERNAL_DIR: C:\Users\uk\Apartment price\data\external
PROCESSED_DIR: C:\Users\uk\Apartment price\data\v2_5year_seoul\processed


In [2]:
# ============================================================
# V2 시장 선행지표 원본 파일 경로 정의
# - 한국부동산원: 매매·전세 수급 및 가격지수
# - 한국은행: 주택가격전망CSI, CCSI
# - 파일 존재 여부를 확인해 경로 오류를 사전에 점검한다.
# ============================================================

DATA_FILES = {
    "sale_supply": (
        EXTERNAL_DIR
        / "(월) 매매수급동향_아파트.csv"
    ),

    "jeonse_supply": (
        EXTERNAL_DIR
        / "(월) 전세수급동향_아파트.csv"
    ),

    "sale_price_index": (
        EXTERNAL_DIR
        / "(월) 매매가격지수_주택종합.csv"
    ),

    "jeonse_price_index": (
        EXTERNAL_DIR
        / "(월) 전세가격지수_주택종합.csv"
    ),

    "housing_csi": (
        EXTERNAL_DIR
        / "주택가격전망CSI와 미분양주택.xlsx"
    ),

    "ccsi": (
        EXTERNAL_DIR
        / "소비자심리지수(CCSI).xlsx"
    ),
}


for name, path in DATA_FILES.items():
    print(
        f"{name:20s}",
        "✅" if path.exists() else "❌",
        path.name,
    )

sale_supply          ✅ (월) 매매수급동향_아파트.csv
jeonse_supply        ✅ (월) 전세수급동향_아파트.csv
sale_price_index     ✅ (월) 매매가격지수_주택종합.csv
jeonse_price_index   ✅ (월) 전세가격지수_주택종합.csv
housing_csi          ✅ 주택가격전망CSI와 미분양주택.xlsx
ccsi                 ✅ 소비자심리지수(CCSI).xlsx


In [3]:
# ============================================================
# R-ONE CSV 자동 로드 함수
# - R-ONE에서 받은 CSV는 파일마다 인코딩이 다를 수 있다.
# - utf-8-sig → cp949 → euc-kr → utf-8 순서로 시도한다.
# - 아직 정제하지 않고 원본 구조를 그대로 읽기 위해 사용한다.
# ============================================================

def read_csv_auto(path, **kwargs):

    encodings = [
        "utf-8-sig",
        "cp949",
        "euc-kr",
        "utf-8",
    ]

    for encoding in encodings:

        try:
            return pd.read_csv(
                path,
                encoding=encoding,
                **kwargs,
            )

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"{path.name} 파일의 인코딩을 확인할 수 없습니다."
    )

In [4]:
# ============================================================
# R-ONE 원본 파일 구조 확인
# - 매매/전세 수급지수와 가격지수의 실제 헤더 구조를 확인한다.
# - R-ONE CSV는 지역명과 연월이 여러 행으로 구성될 수 있으므로
#   정제 함수를 만들기 전에 원본의 상단 행을 먼저 확인한다.
# ============================================================

rone_file_names = [
    "sale_supply",
    "jeonse_supply",
    "sale_price_index",
    "jeonse_price_index",
]

rone_raw = {}


for name in rone_file_names:

    df_raw = read_csv_auto(
        DATA_FILES[name],
        header=None,
    )

    rone_raw[name] = df_raw

    print(f"\n===== {name} =====")
    print("shape:", df_raw.shape)

    display(
        df_raw.head(10)
    )


===== sale_supply =====
shape: (10, 71)


,0,1,2,3,4,5,6,7,8,9,...,61,62,63,64,65,66,67,68,69,70
0,No,지역,지역,지역,2021년 1월,2021년 2월,2021년 3월,2021년 4월,2021년 5월,2021년 6월,...,2025년 10월,2025년 11월,2025년 12월,2026년 1월,2026년 2월,2026년 3월,2026년 4월,2026년 5월,2026년 6월,2026년 7월
1,No,지역,지역,지역,지수,지수,지수,지수,지수,지수,...,지수,지수,지수,지수,지수,지수,지수,지수,지수,지수
2,1,서울,서울,서울,121.7,123.4,115.5,109.0,110.9,121.0,...,104.7,103.0,104.1,106.0,104.7,103.1,105.5,110.2,112.6,114.2
3,2,서울,강북지역,강북지역,126.1,127.4,116.9,107.9,109.3,120.5,...,103.4,101.3,102.9,106.1,105.0,104.2,106.8,112.0,114.1,117.5
4,3,서울,강북지역,도심권,125.0,126.8,112.5,105.4,111.2,123.2,...,111.8,105.1,105.2,105.5,103.0,102.9,103.3,108.2,109.0,110.5
5,4,서울,강북지역,동북권,126.6,128.0,120.8,111.2,111.2,124.5,...,100.6,100.1,101.8,105.7,104.9,104.1,106.7,112.0,115.0,119.3
6,5,서울,강북지역,서북권,125.5,126.2,109.2,100.3,102.7,107.1,...,106.7,102.9,104.9,107.6,106.5,105.1,108.8,114.2,114.3,116.4
7,6,서울,강남지역,강남지역,117.4,119.5,114.2,110.1,112.3,121.5,...,105.9,104.6,105.2,105.9,104.3,102.0,104.3,108.5,111.1,111.0
8,7,서울,강남지역,서남권,117.0,118.5,115.1,110.4,111.0,118.4,...,106.3,104.9,106.2,107.9,106.9,105.8,107.6,110.3,112.7,114.2
9,8,서울,강남지역,동남권,117.9,120.8,113.1,109.7,114.0,125.5,...,105.4,104.2,104.0,103.5,101.0,97.1,100.1,106.2,109.0,106.8



===== jeonse_supply =====
shape: (10, 71)


,0,1,2,3,4,5,6,7,8,9,...,61,62,63,64,65,66,67,68,69,70
0,No,지역,지역,지역,2021년 1월,2021년 2월,2021년 3월,2021년 4월,2021년 5월,2021년 6월,...,2025년 10월,2025년 11월,2025년 12월,2026년 1월,2026년 2월,2026년 3월,2026년 4월,2026년 5월,2026년 6월,2026년 7월
1,No,지역,지역,지역,지수,지수,지수,지수,지수,지수,...,지수,지수,지수,지수,지수,지수,지수,지수,지수,지수
2,1,서울,서울,서울,131.2,126.3,119.5,109.7,110.3,117.2,...,105.0,104.9,105.5,105.5,104.5,105.1,109.5,118.5,126.2,127.5
3,2,서울,강북지역,강북지역,133.5,130.1,122.0,111.4,111.8,118.9,...,105.6,105.2,106.0,107.0,106.1,106.8,110.4,115.7,123.3,123.7
4,3,서울,강북지역,도심권,137.1,130.8,120.5,111.6,103.1,115.6,...,104.9,103.6,103.8,102.3,103.0,105.5,107.9,112.7,116.9,119.2
5,4,서울,강북지역,동북권,133.6,130.4,122.6,110.6,114.6,121.1,...,106.4,106.8,107.7,109.3,107.0,107.3,111.5,116.3,124.7,125.3
6,5,서울,강북지역,서북권,130.3,128.9,121.4,113.6,110.5,115.0,...,103.6,101.6,102.5,103.0,105.0,106.1,108.7,115.9,123.0,121.8
7,6,서울,강남지역,강남지역,129.0,122.5,117.0,108.1,108.8,115.6,...,104.4,104.5,105.0,104.2,102.9,103.5,108.6,121.3,128.9,131.2
8,7,서울,강남지역,서남권,125.8,121.6,117.0,111.8,110.1,112.8,...,104.4,105.0,105.6,105.7,104.6,105.0,108.2,119.5,125.9,128.9
9,8,서울,강남지역,동남권,132.9,123.7,116.9,103.4,107.1,119.2,...,104.4,103.8,104.3,102.1,100.8,101.5,109.3,123.6,132.8,134.1



===== sale_price_index =====
shape: (239, 139)


,0,1,2,3,4,5,6,7,8,9,...,129,130,131,132,133,134,135,136,137,138
0,No,지역,지역,지역,지역,2021년 1월,2021년 1월,2021년 2월,2021년 2월,2021년 3월,...,2026년 3월,2026년 3월,2026년 4월,2026년 4월,2026년 5월,2026년 5월,2026년 6월,2026년 6월,2026년 7월,2026년 7월
1,No,지역,지역,지역,지역,지수,지수,지수,지수,지수,...,지수,지수,지수,지수,지수,지수,지수,지수,지수,지수
2,No,지역,지역,지역,지역,원자료,전기대비증감률,원자료,전기대비증감률,원자료,...,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률
3,1,전국,전국,전국,전국,97.26,0.79,98.13,0.89,98.85,...,99.30,0.15,99.46,0.16,99.67,0.21,100.00,0.33,100.35,0.35
4,2,수도권,수도권,수도권,수도권,92.53,0.80,93.61,1.17,94.51,...,98.57,0.27,98.88,0.31,99.33,0.46,100.00,0.67,100.72,0.72
5,3,지방권,지방권,지방권,지방권,101.93,0.78,102.59,0.64,103.13,...,99.99,0.03,100.02,0.02,99.99,-0.02,100.00,0.01,100.01,0.01
6,4,5대광역시,5대광역시,5대광역시,5대광역시,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.00,100.01,0.01
7,5,4대광역시,4대광역시,4대광역시,4대광역시,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.00,100.01,0.01
8,6,8개도,8개도,8개도,8개도,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.00,100.31,0.31
9,7,7개도,7개도,7개도,7개도,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,100.00,0.00,99.99,-0.01



===== jeonse_price_index =====
shape: (36, 139)


,0,1,2,3,4,5,6,7,8,9,...,129,130,131,132,133,134,135,136,137,138
0,No,지역,지역,지역,지역,2021년 1월,2021년 1월,2021년 2월,2021년 2월,2021년 3월,...,2026년 3월,2026년 3월,2026년 4월,2026년 4월,2026년 5월,2026년 5월,2026년 6월,2026년 6월,2026년 7월,2026년 7월
1,No,지역,지역,지역,지역,지수,지수,지수,지수,지수,...,지수,지수,지수,지수,지수,지수,지수,지수,지수,지수
2,No,지역,지역,지역,지역,원자료,전기대비증감률,원자료,전기대비증감률,원자료,...,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률,원자료,전기대비증감률
3,1,서울,서울,서울,서울,97.87,0.51,98.29,0.42,98.57,...,97.40,0.46,98.05,0.66,98.94,0.91,100.00,1.08,101.03,1.03
4,2,서울,강북지역,강북지역,강북지역,97.22,0.44,97.60,0.39,97.89,...,97.17,0.54,97.85,0.70,98.81,0.98,100.00,1.21,101.13,1.13
5,3,서울,강북지역,도심권,도심권,96.78,0.36,97.07,0.30,97.25,...,97.88,0.46,98.43,0.57,99.16,0.74,100.00,0.85,100.77,0.77
6,4,서울,강북지역,도심권,종로구,97.81,0.21,97.99,0.18,98.10,...,97.98,0.37,98.49,0.53,99.34,0.86,100.00,0.66,100.82,0.82
7,5,서울,강북지역,도심권,중구,99.72,0.30,99.92,0.20,100.01,...,97.65,0.48,98.23,0.59,98.95,0.73,100.00,1.06,100.95,0.95
8,6,서울,강북지역,도심권,용산구,94.75,0.50,95.15,0.42,95.41,...,97.93,0.50,98.50,0.58,99.16,0.67,100.00,0.85,100.65,0.65
9,7,서울,강북지역,동북권,동북권,97.22,0.45,97.63,0.43,97.96,...,96.79,0.55,97.53,0.77,98.62,1.11,100.00,1.40,101.31,1.31


In [5]:
# ============================================================
# 한국은행 Excel 시트 확인
# - 주택가격전망CSI와 CCSI 파일의 시트명을 확인한다.
# - Excel 파일은 데이터 시트 외에 설명/차트 시트가 포함될 수 있으므로
#   실제 데이터가 들어있는 시트를 확인하기 위한 단계다.
# ============================================================

bok_file_names = [
    "housing_csi",
    "ccsi",
]


for name in bok_file_names:

    excel = pd.ExcelFile(
        DATA_FILES[name]
    )

    print(
        f"{name}:",
        excel.sheet_names,
    )

housing_csi: ['Sheet1']
ccsi: ['Sheet1']


In [6]:
# ============================================================
# 한국은행 원본 데이터 구조 확인
# - Excel의 헤더를 임의로 지정하지 않고 원본 형태 그대로 읽는다.
# - 날짜 위치와 실제 CSI 값이 어느 행/열에 있는지 확인한다.
# - 구조 확인 후 두 파일을 월별 테이블로 변환하는 공통 함수를 만든다.
# ============================================================

bok_raw = {}


for name in bok_file_names:

    df_raw = pd.read_excel(
        DATA_FILES[name],
        sheet_name=0,
        header=None,
    )

    bok_raw[name] = df_raw

    print(f"\n===== {name} =====")
    print("shape:", df_raw.shape)

    display(
        df_raw.head(15)
    )


===== housing_csi =====
shape: (315, 3)


,0,1,2
0,주택가격전망CSI와 미분양주택,NaN,NaN
1,주석,1) 주택가격전망CSI는 현재와 비교한 1년 후 전망,NaN
2,출처,"ECOS, \n국토교통부",NaN
3,범례명,주택가격전망CSI(좌축),미분양주택(우축)
4,단위,(Index),(천호)
5,주기,월,월
6,기간,2013-01 ~ 2026-08,2001-01 ~ 2026-06
7,2001-01,NaN,57.284
8,2001-02,NaN,56.142
9,2001-03,NaN,53.269



===== ccsi =====
shape: (225, 2)


,0,1
0,소비자심리지수(CCSI),NaN
1,주석,"1) 장기평균치를 기준값 100으로 하여 100보다 크면 장기평균보다 낙관적임을, ..."
2,출처,ECOS
3,범례명,소비자심리지수(CCSI)
4,단위,(Index)
5,주기,월
6,기간,2008-07 ~ 2026-08
7,2008-07,73.5
8,2008-08,92.1
9,2008-09,90.8


In [7]:
# ============================================================
# V2 시장지표 사용 기간 정의
# - 실제 거래 데이터는 2021-07 ~ 2026-06이지만
#   lag1 Feature 생성을 위해 2021년 상반기 데이터도 유지한다.
# - 최종 모델 입력 시점은 2026-06까지로 제한한다.
# ============================================================

START_MONTH = pd.Period(
    "2021-01",
    freq="M",
)

END_MONTH = pd.Period(
    "2026-06",
    freq="M",
)


print(
    "시장지표 정제 기간:",
    START_MONTH,
    "~",
    END_MONTH,
)

print(
    "총 월 수:",
    len(
        pd.period_range(
            START_MONTH,
            END_MONTH,
            freq="M",
        )
    ),
)

시장지표 정제 기간: 2021-01 ~ 2026-06
총 월 수: 66


In [8]:
# ============================================================
# R-ONE 수급동향 데이터 정제 함수
# - 매매수급동향 / 전세수급동향은 동일한 구조를 가진다.
# - 2021년 1월, 2021년 2월 ... 형태의 Wide 데이터를
#   기준월 × 권역 형태의 Long 데이터로 변환한다.
# - 최종 V2에서는 서울의 5개 생활권 단위 수급지수를 사용한다.
# ============================================================

def clean_supply_index(df_raw, value_name):

    date_cols = list(range(4, df_raw.shape[1]))

    date_map = {
        col: str(df_raw.iloc[0, col]).strip()
        for col in date_cols
    }

    df = df_raw.iloc[2:].copy()

    df = df.rename(
        columns={
            1: "시도",
            2: "대권역",
            3: "권역",
        }
    )

    df = df[
        ["시도", "대권역", "권역"] + date_cols
    ]

    df = df.melt(
        id_vars=["시도", "대권역", "권역"],
        value_vars=date_cols,
        var_name="_date_col",
        value_name=value_name,
    )

    df["기준월"] = (
        df["_date_col"]
        .map(date_map)
        .str.replace("년 ", "-", regex=False)
        .str.replace("월", "", regex=False)
    )

    df["기준월"] = pd.to_datetime(
        df["기준월"],
        format="%Y-%m",
    ).dt.to_period("M")

    df[value_name] = pd.to_numeric(
        df[value_name],
        errors="coerce",
    )

    df = (
        df[
            (df["시도"] == "서울")
            & (df["기준월"] >= START_MONTH)
            & (df["기준월"] <= END_MONTH)
        ]
        .drop(columns="_date_col")
        .reset_index(drop=True)
    )

    return df

In [9]:
# ============================================================
# 매매·전세 수급지수 정제
# - 동일한 정제 함수를 두 데이터에 적용한다.
# - 서울 전체 / 강북·강남지역 / 5개 권역 값이 정상적으로
#   생성되는지 기간과 지역 목록을 확인한다.
# ============================================================

sale_supply = clean_supply_index(
    rone_raw["sale_supply"],
    "매매수급지수",
)

jeonse_supply = clean_supply_index(
    rone_raw["jeonse_supply"],
    "전세수급지수",
)


print("매매수급:", sale_supply.shape)
print("전세수급:", jeonse_supply.shape)

print(
    "\n매매수급 기간:",
    sale_supply["기준월"].min(),
    "~",
    sale_supply["기준월"].max(),
)

print(
    "\n권역:",
    sale_supply["권역"].unique(),
)

display(
    sale_supply.head(10)
)

매매수급: (528, 5)
전세수급: (528, 5)

매매수급 기간: 2021-01 ~ 2026-06

권역: ['서울' '강북지역' '도심권' '동북권' '서북권' '강남지역' '서남권' '동남권']


,시도,대권역,권역,매매수급지수,기준월
0,서울,서울,서울,121.7,2021-01
1,서울,강북지역,강북지역,126.1,2021-01
2,서울,강북지역,도심권,125.0,2021-01
3,서울,강북지역,동북권,126.6,2021-01
4,서울,강북지역,서북권,125.5,2021-01
5,서울,강남지역,강남지역,117.4,2021-01
6,서울,강남지역,서남권,117.0,2021-01
7,서울,강남지역,동남권,117.9,2021-01
8,서울,서울,서울,123.4,2021-02
9,서울,강북지역,강북지역,127.4,2021-02


In [10]:
# ============================================================
# R-ONE 가격지수 데이터 정제 함수
# - 가격지수 파일은 한 달마다
#   '원자료'와 '전기대비증감률' 두 컬럼이 존재한다.
# - 두 값을 Long 형태로 변환한 뒤 월별 한 행으로 합친다.
# - 서울 자치구 단위 데이터만 최종적으로 사용한다.
# ============================================================

def clean_price_index(df_raw, prefix):

    region_cols = [1, 2, 3, 4]
    value_cols = list(range(5, df_raw.shape[1]))

    records = []

    for col in value_cols:

        month = str(df_raw.iloc[0, col]).strip()
        metric = str(df_raw.iloc[2, col]).strip()

        temp = df_raw.iloc[3:, region_cols + [col]].copy()

        temp.columns = [
            "시도",
            "대권역",
            "권역",
            "지역",
            "값",
        ]

        temp["기준월"] = month
        temp["지표구분"] = metric

        records.append(temp)


    df = pd.concat(
        records,
        ignore_index=True,
    )

    df["기준월"] = (
        df["기준월"]
        .str.replace("년 ", "-", regex=False)
        .str.replace("월", "", regex=False)
    )

    df["기준월"] = pd.to_datetime(
        df["기준월"],
        format="%Y-%m",
    ).dt.to_period("M")

    df["값"] = pd.to_numeric(
        df["값"],
        errors="coerce",
    )


    df = df[
        (df["시도"] == "서울")
        & (df["지역"].astype(str).str.endswith("구"))
        & (df["기준월"] >= START_MONTH)
        & (df["기준월"] <= END_MONTH)
    ]


    df = (
        df.pivot_table(
            index=[
                "기준월",
                "지역",
            ],
            columns="지표구분",
            values="값",
            aggfunc="first",
        )
        .reset_index()
        .rename(
            columns={
                "지역": "자치구",
                "원자료": f"{prefix}가격지수",
                "전기대비증감률": f"{prefix}가격지수_전월증감률",
            }
        )
    )

    df.columns.name = None

    return df

In [11]:
# ============================================================
# 매매·전세 가격지수 정제
# - 서울 25개 자치구 × 66개월 구조인지 확인한다.
# - 정상이라면 각각 최대 1,650행(25 × 66)이 생성된다.
# - 가격지수 원자료와 공식 전월 증감률을 모두 보존한다.
# ============================================================

sale_price = clean_price_index(
    rone_raw["sale_price_index"],
    "매매",
)

jeonse_price = clean_price_index(
    rone_raw["jeonse_price_index"],
    "전세",
)


print("매매가격지수:", sale_price.shape)
print("전세가격지수:", jeonse_price.shape)

print(
    "\n매매 자치구 수:",
    sale_price["자치구"].nunique(),
)

print(
    "전세 자치구 수:",
    jeonse_price["자치구"].nunique(),
)

print(
    "\n기간:",
    sale_price["기준월"].min(),
    "~",
    sale_price["기준월"].max(),
)

display(
    sale_price.head(10)
)

매매가격지수: (1650, 4)
전세가격지수: (1650, 4)

매매 자치구 수: 25
전세 자치구 수: 25

기간: 2021-01 ~ 2026-06


,기준월,자치구,매매가격지수,매매가격지수_전월증감률
0,2021-01,강남구,79.49,0.56
1,2021-01,강동구,83.62,0.66
2,2021-01,강북구,97.18,0.38
3,2021-01,강서구,94.11,0.29
4,2021-01,관악구,93.64,0.35
5,2021-01,광진구,85.25,0.35
6,2021-01,구로구,93.62,0.23
7,2021-01,금천구,99.72,0.22
8,2021-01,노원구,97.08,0.45
9,2021-01,도봉구,100.66,0.39


In [12]:
# ============================================================
# 주택가격전망CSI 정제
# - Excel 7행부터 실제 월별 데이터가 시작된다.
# - 두 번째 컬럼의 주택가격전망CSI만 사용한다.
# - 함께 제공된 미분양주택 데이터는 V2 확장 후보로 보관한다.
# ============================================================

housing_csi = (
    bok_raw["housing_csi"]
    .iloc[7:, [0, 1]]
    .copy()
)

housing_csi.columns = [
    "기준월",
    "주택가격전망CSI",
]

housing_csi["기준월"] = pd.to_datetime(
    housing_csi["기준월"],
    format="%Y-%m",
    errors="coerce",
).dt.to_period("M")

housing_csi["주택가격전망CSI"] = pd.to_numeric(
    housing_csi["주택가격전망CSI"],
    errors="coerce",
)

housing_csi = (
    housing_csi[
        (housing_csi["기준월"] >= START_MONTH)
        & (housing_csi["기준월"] <= END_MONTH)
    ]
    .dropna()
    .reset_index(drop=True)
)

print("CSI:", housing_csi.shape)
display(housing_csi.head())
display(housing_csi.tail())

CSI: (66, 2)


,기준월,주택가격전망CSI
0,2021-01,130.0
1,2021-02,129.0
2,2021-03,124.0
3,2021-04,122.0
4,2021-05,124.0


,기준월,주택가격전망CSI
61,2026-02,108.0
62,2026-03,96.0
63,2026-04,104.0
64,2026-05,112.0
65,2026-06,120.0


In [13]:
# ============================================================
# 소비자심리지수(CCSI) 정제
# - Excel 7행부터 월별 실제 데이터가 시작된다.
# - 전국 단위 월별 심리지표이므로 지역 구분 없이 기준월만 유지한다.
# - 향후 모든 서울 자치구에 같은 월의 CCSI가 연결된다.
# ============================================================

ccsi = (
    bok_raw["ccsi"]
    .iloc[7:, [0, 1]]
    .copy()
)

ccsi.columns = [
    "기준월",
    "CCSI",
]

ccsi["기준월"] = pd.to_datetime(
    ccsi["기준월"],
    format="%Y-%m",
    errors="coerce",
).dt.to_period("M")

ccsi["CCSI"] = pd.to_numeric(
    ccsi["CCSI"],
    errors="coerce",
)

ccsi = (
    ccsi[
        (ccsi["기준월"] >= START_MONTH)
        & (ccsi["기준월"] <= END_MONTH)
    ]
    .dropna()
    .reset_index(drop=True)
)

print("CCSI:", ccsi.shape)
display(ccsi.head())
display(ccsi.tail())

CCSI: (66, 2)


,기준월,CCSI
0,2021-01,95.0
1,2021-02,97.2
2,2021-03,100.4
3,2021-04,102.3
4,2021-05,105.5


,기준월,CCSI
61,2026-02,112.1
62,2026-03,107.0
63,2026-04,99.2
64,2026-05,106.1
65,2026-06,106.6


In [14]:
# ============================================================
# 서울 25개 자치구 → R-ONE 5개 권역 매핑
# - 가격지수는 자치구 단위지만 수급지수는 5개 권역 단위다.
# - 각 자치구에 해당 권역의 매매·전세 수급지수를 연결하기 위해 사용한다.
# - 25개 자치구가 모두 정확히 하나의 권역에 속하는지 함께 확인한다.
# ============================================================

SEOUL_REGION_MAP = {
    # 도심권
    "종로구": "도심권",
    "중구": "도심권",
    "용산구": "도심권",

    # 동북권
    "성동구": "동북권",
    "광진구": "동북권",
    "동대문구": "동북권",
    "중랑구": "동북권",
    "성북구": "동북권",
    "강북구": "동북권",
    "도봉구": "동북권",
    "노원구": "동북권",

    # 서북권
    "은평구": "서북권",
    "서대문구": "서북권",
    "마포구": "서북권",

    # 서남권
    "양천구": "서남권",
    "강서구": "서남권",
    "구로구": "서남권",
    "금천구": "서남권",
    "영등포구": "서남권",
    "동작구": "서남권",
    "관악구": "서남권",

    # 동남권
    "서초구": "동남권",
    "강남구": "동남권",
    "송파구": "동남권",
    "강동구": "동남권",
}


print("매핑 자치구 수:", len(SEOUL_REGION_MAP))

display(
    pd.Series(SEOUL_REGION_MAP)
    .value_counts()
    .rename("자치구수")
    .to_frame()
)

매핑 자치구 수: 25


,자치구수
동북권,8
서남권,7
동남권,4
도심권,3
서북권,3


In [15]:
# ============================================================
# 자치구 연결용 5개 권역 수급지수 추출
# - 서울 전체 / 강북지역 / 강남지역 집계값은 제외한다.
# - 도심권·동북권·서북권·서남권·동남권만 사용한다.
# - 매매와 전세 수급지수를 기준월 + 권역 기준으로 병합한다.
# ============================================================

SEOUL_REGIONS = [
    "도심권",
    "동북권",
    "서북권",
    "서남권",
    "동남권",
]


sale_supply_region = (
    sale_supply[
        sale_supply["권역"].isin(SEOUL_REGIONS)
    ][
        [
            "기준월",
            "권역",
            "매매수급지수",
        ]
    ]
    .copy()
)


jeonse_supply_region = (
    jeonse_supply[
        jeonse_supply["권역"].isin(SEOUL_REGIONS)
    ][
        [
            "기준월",
            "권역",
            "전세수급지수",
        ]
    ]
    .copy()
)


supply_market = (
    sale_supply_region
    .merge(
        jeonse_supply_region,
        on=["기준월", "권역"],
        how="inner",
        validate="one_to_one",
    )
)


print("수급지수:", supply_market.shape)
print("권역 수:", supply_market["권역"].nunique())
print("월 수:", supply_market["기준월"].nunique())

display(supply_market.head())

수급지수: (330, 4)
권역 수: 5
월 수: 66


,기준월,권역,매매수급지수,전세수급지수
0,2021-01,도심권,125.0,137.1
1,2021-01,동북권,126.6,133.6
2,2021-01,서북권,125.5,130.3
3,2021-01,서남권,117.0,125.8
4,2021-01,동남권,117.9,132.9


In [16]:
# ============================================================
# 자치구별 매매·전세 가격지수 통합
# - 매매가격지수와 전세가격지수는 모두 25개 자치구 × 66개월 구조다.
# - 기준월 + 자치구를 Key로 병합한다.
# - 병합 후에도 1,650행이 유지되어야 한다.
# ============================================================

price_market = (
    sale_price
    .merge(
        jeonse_price,
        on=["기준월", "자치구"],
        how="inner",
        validate="one_to_one",
    )
)


print("가격지수:", price_market.shape)
print("자치구 수:", price_market["자치구"].nunique())
print("월 수:", price_market["기준월"].nunique())

display(price_market.head())

가격지수: (1650, 6)
자치구 수: 25
월 수: 66


,기준월,자치구,매매가격지수,매매가격지수_전월증감률,전세가격지수,전세가격지수_전월증감률
0,2021-01,강남구,79.49,0.56,103.12,0.82
1,2021-01,강동구,83.62,0.66,95.14,0.68
2,2021-01,강북구,97.18,0.38,99.71,0.46
3,2021-01,강서구,94.11,0.29,101.69,0.36
4,2021-01,관악구,93.64,0.35,104.27,0.40


In [17]:
# ============================================================
# 자치구별 가격지수에 R-ONE 권역 추가
# - 자치구별 가격 테이블에 앞서 정의한 5개 권역을 연결한다.
# - 이후 기준월 + 권역으로 수급지수를 붙이기 위한 준비 단계다.
# - 매핑되지 않은 자치구가 있으면 반드시 확인한다.
# ============================================================

price_market["권역"] = (
    price_market["자치구"]
    .map(SEOUL_REGION_MAP)
)


print(
    "권역 매핑 결측:",
    price_market["권역"].isna().sum(),
)

display(
    price_market[
        [
            "자치구",
            "권역",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["권역", "자치구"]
    )
)

권역 매핑 결측: 0


,자치구,권역
20,용산구,도심권
22,종로구,도심권
23,중구,도심권
0,강남구,동남권
1,강동구,동남권
14,서초구,동남권
17,송파구,동남권
2,강북구,동북권
5,광진구,동북권
8,노원구,동북권


In [18]:
# ============================================================
# 자치구 가격지수 + 권역 수급지수 통합
# - 가격지수는 자치구 단위, 수급지수는 5개 권역 단위다.
# - 각 자치구에 해당 권역의 같은 월 수급지수를 연결한다.
# - 최종 행 수는 25개 자치구 × 66개월 = 1,650행을 유지한다.
# ============================================================

market_v2 = (
    price_market
    .merge(
        supply_market,
        on=["기준월", "권역"],
        how="left",
        validate="many_to_one",
    )
)


print("가격 + 수급:", market_v2.shape)

print("\n결측치:")
print(
    market_v2[
        [
            "매매가격지수",
            "전세가격지수",
            "매매수급지수",
            "전세수급지수",
        ]
    ]
    .isna()
    .sum()
)

display(market_v2.head())

가격 + 수급: (1650, 9)

결측치:
매매가격지수    0
전세가격지수    0
매매수급지수    0
전세수급지수    0
dtype: int64


,기준월,자치구,매매가격지수,매매가격지수_전월증감률,전세가격지수,전세가격지수_전월증감률,권역,매매수급지수,전세수급지수
0,2021-01,강남구,79.49,0.56,103.12,0.82,동남권,117.9,132.9
1,2021-01,강동구,83.62,0.66,95.14,0.68,동남권,117.9,132.9
2,2021-01,강북구,97.18,0.38,99.71,0.46,동북권,126.6,133.6
3,2021-01,강서구,94.11,0.29,101.69,0.36,서남권,117.0,125.8
4,2021-01,관악구,93.64,0.35,104.27,0.40,서남권,117.0,125.8


In [19]:
# ============================================================
# 한국은행 심리지표 통합
# - 주택가격전망CSI와 CCSI는 모두 전국 월별 지표다.
# - 기준월을 Key로 하나의 심리 테이블로 결합한다.
# - 정상이라면 2021-01 ~ 2026-06 총 66행이 유지된다.
# ============================================================

sentiment_market = (
    housing_csi
    .merge(
        ccsi,
        on="기준월",
        how="inner",
        validate="one_to_one",
    )
)


print("심리지표:", sentiment_market.shape)

print("\n결측치:")
print(
    sentiment_market[
        [
            "주택가격전망CSI",
            "CCSI",
        ]
    ]
    .isna()
    .sum()
)

display(sentiment_market.head())
display(sentiment_market.tail())

심리지표: (66, 3)

결측치:
주택가격전망CSI    0
CCSI         0
dtype: int64


,기준월,주택가격전망CSI,CCSI
0,2021-01,130.0,95.0
1,2021-02,129.0,97.2
2,2021-03,124.0,100.4
3,2021-04,122.0,102.3
4,2021-05,124.0,105.5


,기준월,주택가격전망CSI,CCSI
61,2026-02,108.0,112.1
62,2026-03,96.0,107.0
63,2026-04,104.0,99.2
64,2026-05,112.0,106.1
65,2026-06,120.0,106.6


In [20]:
# ============================================================
# V2 시장지표 원자료 최종 통합
# - 자치구별 가격지수 + 권역별 수급지수 + 전국 심리지표를 결합한다.
# - 아직 lag나 변화량을 만들지 않은 '원자료 기준 테이블'이다.
# - 정상이라면 25개 자치구 × 66개월 = 1,650행이어야 한다.
# ============================================================

market_v2 = (
    market_v2
    .merge(
        sentiment_market,
        on="기준월",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        ["자치구", "기준월"]
    )
    .reset_index(drop=True)
)


print("최종 시장 원자료:", market_v2.shape)
print("자치구 수:", market_v2["자치구"].nunique())
print("월 수:", market_v2["기준월"].nunique())

display(market_v2.head(10))

최종 시장 원자료: (1650, 11)
자치구 수: 25
월 수: 66


,기준월,자치구,매매가격지수,매매가격지수_전월증감률,전세가격지수,전세가격지수_전월증감률,권역,매매수급지수,전세수급지수,주택가격전망CSI,CCSI
0,2021-01,강남구,79.49,0.56,103.12,0.82,동남권,117.9,132.9,130.0,95.0
1,2021-02,강남구,79.95,0.57,103.80,0.66,동남권,120.8,123.7,129.0,97.2
2,2021-03,강남구,80.31,0.45,104.15,0.33,동남권,113.1,116.9,124.0,100.4
3,2021-04,강남구,80.71,0.50,104.14,-0.01,동남권,109.7,103.4,122.0,102.3
4,2021-05,강남구,81.19,0.60,104.15,0.01,동남권,114.0,107.1,124.0,105.5
5,2021-06,강남구,82.06,1.06,104.59,0.42,동남권,125.5,119.2,127.0,110.9
6,2021-07,강남구,82.67,0.75,105.11,0.50,동남권,127.0,120.5,129.0,103.4
7,2021-08,강남구,83.33,0.80,105.70,0.56,동남권,127.3,119.2,129.0,102.7
8,2021-09,강남구,84.35,1.22,106.52,0.78,동남권,127.3,113.8,128.0,104.1
9,2021-10,강남구,85.27,1.09,107.28,0.71,동남권,117.2,103.9,125.0,107.3


In [21]:
# ============================================================
# V2 시장지표 원자료 무결성 검증
# - 기대 행 수: 25개 자치구 × 66개월 = 1,650
# - 기준월 + 자치구 중복이 없어야 한다.
# - 핵심 시장지표에 결측이 없어야 한다.
# - 모든 조건이 만족되어야 다음 Feature Engineering으로 진행한다.
# ============================================================

EXPECTED_ROWS = 25 * 66

key_duplicates = market_v2.duplicated(
    subset=["기준월", "자치구"]
).sum()

check_cols = [
    "매매가격지수",
    "전세가격지수",
    "매매수급지수",
    "전세수급지수",
    "주택가격전망CSI",
    "CCSI",
]


print("행 수:", len(market_v2), "/", EXPECTED_ROWS)
print("Key 중복:", key_duplicates)

print("\n핵심 Feature 결측:")
print(
    market_v2[check_cols]
    .isna()
    .sum()
)


assert len(market_v2) == EXPECTED_ROWS
assert key_duplicates == 0
assert market_v2[check_cols].isna().sum().sum() == 0


print("\n✅ 시장지표 원자료 무결성 검증 완료")

행 수: 1650 / 1650
Key 중복: 0

핵심 Feature 결측:
매매가격지수       0
전세가격지수       0
매매수급지수       0
전세수급지수       0
주택가격전망CSI    0
CCSI         0
dtype: int64

✅ 시장지표 원자료 무결성 검증 완료


In [22]:
# ============================================================
# 선행 Feature 생성을 위한 데이터 정렬
# - Lag과 변화량은 시간 순서가 정확해야 하므로 자치구별 기준월로 정렬한다.
# - 이후 모든 Feature는 최소 1개월 이전 정보만 이용한다.
# - 현재월 시장지표는 모델 입력 Feature로 직접 사용하지 않는다.
# ============================================================

market_features_v2 = (
    market_v2
    .sort_values(
        ["자치구", "기준월"]
    )
    .reset_index(drop=True)
    .copy()
)

print(
    "정렬 완료:",
    market_features_v2.shape,
)

display(
    market_features_v2[
        [
            "기준월",
            "자치구",
            "권역",
        ]
    ]
    .head()
)

정렬 완료: (1650, 11)


,기준월,자치구,권역
0,2021-01,강남구,동남권
1,2021-02,강남구,동남권
2,2021-03,강남구,동남권
3,2021-04,강남구,동남권
4,2021-05,강남구,동남권


In [23]:
# ============================================================
# 핵심 시장지표 Lag1 생성
# - 거래가 발생한 현재월의 지표 대신 전월 값을 사용한다.
# - 발표 시차로 인한 미래정보 누출을 보수적으로 방지한다.
# - 가격·수급·심리의 현재 상태를 나타내는 기본 V2 Feature다.
# ============================================================

CORE_MARKET_COLS = [
    "매매가격지수",
    "전세가격지수",
    "매매수급지수",
    "전세수급지수",
    "주택가격전망CSI",
    "CCSI",
]


for col in CORE_MARKET_COLS:

    market_features_v2[f"{col}_lag1"] = (
        market_features_v2
        .groupby("자치구")[col]
        .shift(1)
    )


lag1_cols = [
    f"{col}_lag1"
    for col in CORE_MARKET_COLS
]


display(
    market_features_v2[
        [
            "기준월",
            "자치구",
        ]
        + lag1_cols
    ]
    .head(8)
)

,기준월,자치구,매매가격지수_lag1,전세가격지수_lag1,매매수급지수_lag1,전세수급지수_lag1,주택가격전망CSI_lag1,CCSI_lag1
0,2021-01,강남구,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-02,강남구,79.49,103.12,117.9,132.9,130.0,95.0
2,2021-03,강남구,79.95,103.80,120.8,123.7,129.0,97.2
3,2021-04,강남구,80.31,104.15,113.1,116.9,124.0,100.4
4,2021-05,강남구,80.71,104.14,109.7,103.4,122.0,102.3
5,2021-06,강남구,81.19,104.15,114.0,107.1,124.0,105.5
6,2021-07,강남구,82.06,104.59,125.5,119.2,127.0,110.9
7,2021-08,강남구,82.67,105.11,127.0,120.5,129.0,103.4


In [24]:
# ============================================================
# 전월 기준 1개월 변화량 생성
# - t-1과 t-2를 비교해 최근 한 달의 방향성을 나타낸다.
# - 현재월(t)의 값은 사용하지 않는다.
# - 수급과 심리지표의 단기 변화가 상승/하락 전환 예측에
#   도움이 되는지 V2에서 검증한다.
# ============================================================

CHANGE_COLS = [
    "매매수급지수",
    "전세수급지수",
    "주택가격전망CSI",
    "CCSI",
]


for col in CHANGE_COLS:

    grouped = (
        market_features_v2
        .groupby("자치구")[col]
    )

    market_features_v2[f"{col}_diff1"] = (
        grouped.shift(1)
        - grouped.shift(2)
    )

In [25]:
# ============================================================
# 가격지수 공식 전월 증감률 Lag1 생성
# - R-ONE이 제공하는 매매·전세 가격지수 전기대비증감률을 활용한다.
# - 거래월 직전까지 공개된 가격 움직임만 사용하기 위해 1개월 Lag한다.
# - 직접 계산한 변화율 대신 공식 통계의 증감률을 보존한다.
# ============================================================

PRICE_CHANGE_COLS = [
    "매매가격지수_전월증감률",
    "전세가격지수_전월증감률",
]


for col in PRICE_CHANGE_COLS:

    market_features_v2[f"{col}_lag1"] = (
        market_features_v2
        .groupby("자치구")[col]
        .shift(1)
    )

In [26]:
# ============================================================
# 가격지수 3개월 Momentum 생성
# - 현재월을 제외하고 t-1 대비 t-4의 가격지수 변화율을 계산한다.
# - 단기 한 달 움직임보다 시장의 지속적인 상승·하락 흐름을 포착한다.
# - 미래 3개월 상승 여부를 예측하는 Classification V2에서
#   중요한 선행 Feature 후보로 사용한다.
# ============================================================

PRICE_INDEX_COLS = [
    "매매가격지수",
    "전세가격지수",
]


for col in PRICE_INDEX_COLS:

    grouped = (
        market_features_v2
        .groupby("자치구")[col]
    )

    lag1 = grouped.shift(1)
    lag4 = grouped.shift(4)

    market_features_v2[f"{col}_momentum3m"] = (
        (lag1 / lag4 - 1)
        * 100
    )

In [27]:
# ============================================================
# 수급·심리지표 3개월 변화량 생성
# - t-1과 t-4를 비교해 최근 3개월의 방향성을 측정한다.
# - 절대 수준(lag1)과 함께 사용해
#   '현재 상태'와 '최근 변화 방향'을 동시에 표현한다.
# ============================================================

for col in CHANGE_COLS:

    grouped = (
        market_features_v2
        .groupby("자치구")[col]
    )

    market_features_v2[f"{col}_diff3m"] = (
        grouped.shift(1)
        - grouped.shift(4)
    )

In [28]:
# ============================================================
# 가격지수 수준값 미래정보 누수 점검
# - R-ONE 가격지수는 최신 기준월(2026-06)=100으로 리베이스되어 제공된다.
# - 과거 시점의 지수 수준은 (해당 월 가격 / 2026-06 가격) × 100이 되어
#   이후 상승폭이라는 미래정보를 담는다. (예: 2021-06 송파구 75.61 → 이후 약 +32%)
# - 증감률·Momentum은 기준시점이 바뀌어도 값이 같으므로 안전하다.
# - 따라서 매매·전세 가격지수 수준값(lag1)은 최종 Feature에서 제외한다.
# ============================================================

base_month_check = (
    market_features_v2[
        market_features_v2["기준월"] == END_MONTH
    ][PRICE_INDEX_COLS]
)

print(f"{END_MONTH} 가격지수 (전 자치구):")
print(
    base_month_check
    .agg(["min", "max"])
)


assert (
    base_month_check
    == 100
).all().all()


print(
    f"\n⚠️ 지수 기준시점 = {END_MONTH} 확인 → 가격지수 수준값은 Feature에서 제외"
)

2026-06 가격지수 (전 자치구):
     매매가격지수  전세가격지수
min   100.0   100.0
max   100.0   100.0

⚠️ 지수 기준시점 = 2026-06 확인 → 가격지수 수준값은 Feature에서 제외


In [29]:
# ============================================================
# V2 시장 선행 Feature 목록 정의
# - 모델에 실제 사용할 시장 Feature만 명시적으로 관리한다.
# - 현재월 원자료는 제외하고 과거 시점에서 계산된 Feature만 포함한다.
# - 가격지수는 수준값을 제외하고 기준시점과 무관한 증감률·Momentum만 사용한다.
# - 이후 Regression / Classification에서 동일한 목록을 재사용한다.
# ============================================================

MARKET_V2_FEATURES = [

    # 전월 수준
    # - 매매·전세 가격지수 수준값(lag1)은 2026-06=100 리베이스로
    #   미래 상승폭 정보를 포함하므로 제외한다. (위 누수 점검 셀 참고)
    "매매수급지수_lag1",
    "전세수급지수_lag1",
    "주택가격전망CSI_lag1",
    "CCSI_lag1",

    # 1개월 변화
    "매매수급지수_diff1",
    "전세수급지수_diff1",
    "주택가격전망CSI_diff1",
    "CCSI_diff1",

    # 공식 가격 증감률
    "매매가격지수_전월증감률_lag1",
    "전세가격지수_전월증감률_lag1",

    # 3개월 흐름
    "매매가격지수_momentum3m",
    "전세가격지수_momentum3m",
    "매매수급지수_diff3m",
    "전세수급지수_diff3m",
    "주택가격전망CSI_diff3m",
    "CCSI_diff3m",
]


print(
    "V2 시장 Feature 수:",
    len(MARKET_V2_FEATURES),
)

print()

for col in MARKET_V2_FEATURES:
    print("-", col)

V2 시장 Feature 수: 16

- 매매수급지수_lag1
- 전세수급지수_lag1
- 주택가격전망CSI_lag1
- CCSI_lag1
- 매매수급지수_diff1
- 전세수급지수_diff1
- 주택가격전망CSI_diff1
- CCSI_diff1
- 매매가격지수_전월증감률_lag1
- 전세가격지수_전월증감률_lag1
- 매매가격지수_momentum3m
- 전세가격지수_momentum3m
- 매매수급지수_diff3m
- 전세수급지수_diff3m
- 주택가격전망CSI_diff3m
- CCSI_diff3m


In [30]:
# ============================================================
# V2 선행 Feature 결측 검증
# - Lag 및 3개월 Momentum 때문에 2021년 초반에는 결측이 발생한다.
# - 실제 거래 데이터 시작점인 2021-07 이후에는
#   모든 V2 시장 Feature가 완전하게 존재해야 한다.
# ============================================================

model_period_market = (
    market_features_v2[
        market_features_v2["기준월"]
        >= pd.Period("2021-07", freq="M")
    ]
)


missing_check = (
    model_period_market[
        MARKET_V2_FEATURES
    ]
    .isna()
    .sum()
)


print(
    "2021-07 이후 행 수:",
    len(model_period_market),
)

print("\nFeature 결측:")
print(missing_check)


assert (
    missing_check.sum()
    == 0
)


print(
    "\n✅ 실제 모델링 기간의 시장 선행 Feature 결측 없음"
)

2021-07 이후 행 수: 1500

Feature 결측:
매매수급지수_lag1          0
전세수급지수_lag1          0
주택가격전망CSI_lag1       0
CCSI_lag1            0
매매수급지수_diff1         0
전세수급지수_diff1         0
주택가격전망CSI_diff1      0
CCSI_diff1           0
매매가격지수_전월증감률_lag1    0
전세가격지수_전월증감률_lag1    0
매매가격지수_momentum3m    0
전세가격지수_momentum3m    0
매매수급지수_diff3m        0
전세수급지수_diff3m        0
주택가격전망CSI_diff3m     0
CCSI_diff3m          0
dtype: int64

✅ 실제 모델링 기간의 시장 선행 Feature 결측 없음


In [31]:
# ============================================================
# V2 시장 선행지표 최종 테이블 생성
# - 실제 모델링 기간인 2021-07 ~ 2026-06만 사용한다.
# - 현재월 원자료는 제외하고 과거 정보로 생성한 Feature만 저장한다.
# - 병합 Key인 기준월·자치구와 수급지수 해석용 권역을 함께 유지한다.
# ============================================================

market_leading_v2 = (
    market_features_v2[
        market_features_v2["기준월"]
        >= pd.Period("2021-07", freq="M")
    ][
        [
            "기준월",
            "자치구",
            "권역",
        ]
        + MARKET_V2_FEATURES
    ]
    .copy()
    .reset_index(drop=True)
)


print("최종 시장 선행지표:", market_leading_v2.shape)
print("자치구 수:", market_leading_v2["자치구"].nunique())
print("월 수:", market_leading_v2["기준월"].nunique())
print("Feature 수:", len(MARKET_V2_FEATURES))

display(market_leading_v2.head())

최종 시장 선행지표: (1500, 19)
자치구 수: 25
월 수: 60
Feature 수: 16


,기준월,자치구,권역,매매수급지수_lag1,전세수급지수_lag1,주택가격전망CSI_lag1,CCSI_lag1,매매수급지수_diff1,전세수급지수_diff1,주택가격전망CSI_diff1,CCSI_diff1,매매가격지수_전월증감률_lag1,전세가격지수_전월증감률_lag1,매매가격지수_momentum3m,전세가격지수_momentum3m,매매수급지수_diff3m,전세수급지수_diff3m,주택가격전망CSI_diff3m,CCSI_diff3m
0,2021-07,강남구,동남권,125.5,119.2,127.0,110.9,11.5,12.1,3.0,5.4,1.06,0.42,2.179056,0.422468,12.4,2.3,3.0,10.5
1,2021-08,강남구,동남권,127.0,120.5,129.0,103.4,1.5,1.3,2.0,-7.5,0.75,0.50,2.428448,0.931438,17.3,17.1,7.0,1.1
2,2021-09,강남구,동남권,127.3,119.2,129.0,102.7,0.3,-1.3,0.0,-0.7,0.80,0.56,2.635793,1.488238,13.3,12.1,5.0,-2.8
3,2021-10,강남구,동남권,127.3,113.8,128.0,104.1,0.0,-5.4,-1.0,1.4,1.22,0.78,2.790641,1.845301,1.8,-5.4,1.0,-6.8
4,2021-11,강남구,동남권,117.2,103.9,125.0,107.3,-10.1,-9.9,-3.0,3.2,1.09,0.71,3.145034,2.064504,-9.8,-16.6,-4.0,3.9


In [32]:
# ============================================================
# 시장 선행지표 저장 전 최종 검증
# - 기준월 + 자치구 조합이 유일한지 확인한다.
# - 모델 입력 Feature에 결측치가 없는지 확인한다.
# - 현재월 원자료가 실수로 포함되지 않았는지 컬럼 구조를 점검한다.
# ============================================================

duplicate_count = market_leading_v2.duplicated(
    subset=["기준월", "자치구"]
).sum()

missing_count = (
    market_leading_v2[MARKET_V2_FEATURES]
    .isna()
    .sum()
    .sum()
)


print("행 수:", len(market_leading_v2))
print("Key 중복:", duplicate_count)
print("Feature 전체 결측:", missing_count)

print("\n최종 컬럼:")
print(market_leading_v2.columns.tolist())


assert len(market_leading_v2) == 1500
assert duplicate_count == 0
assert missing_count == 0


print("\n✅ 시장 선행지표 최종 검증 완료")

행 수:

 1500
Key 중복: 0
Feature 전체 결측: 0

최종 컬럼:
['기준월', '자치구', '권역', '매매수급지수_lag1', '전세수급지수_lag1', '주택가격전망CSI_lag1', 'CCSI_lag1', '매매수급지수_diff1', '전세수급지수_diff1', '주택가격전망CSI_diff1', 'CCSI_diff1', '매매가격지수_전월증감률_lag1', '전세가격지수_전월증감률_lag1', '매매가격지수_momentum3m', '전세가격지수_momentum3m', '매매수급지수_diff3m', '전세수급지수_diff3m', '주택가격전망CSI_diff3m', 'CCSI_diff3m']

✅ 시장 선행지표 최종 검증 완료


In [33]:
# ============================================================
# V2 시장 선행지표 Parquet 저장
# - 기준월은 환경 호환성을 위해 YYYY-MM 문자열로 저장한다.
# - V1 원본과 분리하여 market_leading_v2.parquet으로 관리한다.
# - 이후 V2 Feature Merge 노트북에서 이 파일을 불러와 사용한다.
# ============================================================

market_save = market_leading_v2.copy()

market_save["기준월"] = (
    market_save["기준월"]
    .astype(str)
)


MARKET_OUTPUT_PATH = (
    PROCESSED_DIR
    / "market_leading_v2.parquet"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

market_save.to_parquet(
    MARKET_OUTPUT_PATH,
    index=False,
)


print("저장 완료:")
print(MARKET_OUTPUT_PATH)

print(
    "\n파일 존재:",
    MARKET_OUTPUT_PATH.exists(),
)

저장 완료:
C:\Users\uk\Apartment price\data\v2_5year_seoul\processed\market_leading_v2.parquet

파일 존재: True


In [34]:
# ============================================================
# 저장된 시장 선행지표 재로드 검증
# - 실제 저장된 Parquet 파일을 다시 읽어 데이터 손상 여부를 확인한다.
# - 행 수, 컬럼 수, Key 중복, 결측치를 다시 점검한다.
# ============================================================

market_check = pd.read_parquet(
    MARKET_OUTPUT_PATH
)


print("shape:", market_check.shape)

print(
    "Key 중복:",
    market_check.duplicated(
        subset=["기준월", "자치구"]
    ).sum(),
)

print(
    "Feature 결측:",
    market_check[
        MARKET_V2_FEATURES
    ]
    .isna()
    .sum()
    .sum(),
)

display(market_check.head())

shape:

 (1500, 19)
Key 중복: 0
Feature 결측: 0


,기준월,자치구,권역,매매수급지수_lag1,전세수급지수_lag1,주택가격전망CSI_lag1,CCSI_lag1,매매수급지수_diff1,전세수급지수_diff1,주택가격전망CSI_diff1,CCSI_diff1,매매가격지수_전월증감률_lag1,전세가격지수_전월증감률_lag1,매매가격지수_momentum3m,전세가격지수_momentum3m,매매수급지수_diff3m,전세수급지수_diff3m,주택가격전망CSI_diff3m,CCSI_diff3m
0,2021-07,강남구,동남권,125.5,119.2,127.0,110.9,11.5,12.1,3.0,5.4,1.06,0.42,2.179056,0.422468,12.4,2.3,3.0,10.5
1,2021-08,강남구,동남권,127.0,120.5,129.0,103.4,1.5,1.3,2.0,-7.5,0.75,0.50,2.428448,0.931438,17.3,17.1,7.0,1.1
2,2021-09,강남구,동남권,127.3,119.2,129.0,102.7,0.3,-1.3,0.0,-0.7,0.80,0.56,2.635793,1.488238,13.3,12.1,5.0,-2.8
3,2021-10,강남구,동남권,127.3,113.8,128.0,104.1,0.0,-5.4,-1.0,1.4,1.22,0.78,2.790641,1.845301,1.8,-5.4,1.0,-6.8
4,2021-11,강남구,동남권,117.2,103.9,125.0,107.3,-10.1,-9.9,-3.0,3.2,1.09,0.71,3.145034,2.064504,-9.8,-16.6,-4.0,3.9
